## Defacing Pipeline with `caideface`

This notebook demonstrates how to use the [`caideface`](https://pypi.org/project/caideface/) Python package to deface head MRI and CT scans. The MRI pipeline is described in *"A Generalisable Head MRI Defacing Pipeline: Evaluation on 2,566 Meningioma Scans"* ([arXiv:2505.12999](https://arxiv.org/abs/2505.12999)).

![Pipeline Overview](../Pipeline_smaller.svg)

*Figure 1: Overview of the defacing pipeline. Steps are shown in blue; the rhomboid illustrates voxel orientation before and after reorientation to MNI152 space.*

The pipeline consists of three steps:

1. **Reorientation** — Aligns scans to RAS canonical orientation (MNI152 standard).
2. **Skull-stripping** — Extracts a brain mask using [HD-BET](https://github.com/MIC-DKFZ/HD-BET) (MRI) or [TotalSegmentator](https://github.com/wasserth/TotalSegmentator) (CT), then dilates it (default 14 mm) to preserve peripheral brain structures.
3. **Registration & Defacing** — Registers each scan to a modality-matched template via affine registration ([BRAINSFit](https://github.com/BRAINSia/BRAINSTools)), warps the template face mask into the scan's space, and applies it to remove facial features.

This notebook shows two ways to run the pipeline:
- **CLI** — shell commands via `!caideface ...`
- **Python API** — using `caideface` functions directly

### Getting Started

Before running this notebook, set up a conda environment with `caideface` and register it as a Jupyter kernel. Run the following commands in your **terminal**:

```bash
# 1. Create a conda environment
conda create -n caideface python=3.10 -y
conda activate caideface

# 2. Install caideface (use caideface[ct] for CT defacing support)
pip install caideface

# 3. Register the environment as a Jupyter kernel
pip install ipykernel
python -m ipykernel install --user --name caideface --display-name "Python (caideface)"
```

Then open this notebook and select **Python (caideface)** as the kernel (top-right in VS Code, or **Kernel > Change Kernel** in Jupyter).

> **Note:** If you are working with a development version of `caideface`, install from the local source instead: `pip install -e ../caideface`

### Configuration

Set the paths to your input data, output directory, and the BRAINSFit/BRAINSResample executables. These are bundled with [3D Slicer](https://www.slicer.org/) — see the [main README](../README.md#extracting-brainsfit-and-brainsresample-from-3d-slicer) for how to locate them.

In [ ]:
import os

# --- Paths (adjust these to your setup) ---
INPUT_DIR = os.path.abspath("../data/example_input_images")
OUTPUT_DIR = os.path.abspath("../data/example_output_caideface")
MODALITY = "mri"  # "mri" or "ct"

# BRAINSFit / BRAINSResample executables (bundled with 3D Slicer)
# e.g.:
#   macOS:  /Applications/Slicer.app/Contents/lib/Slicer-5.8/cli-modules/BRAINSFit
#   Linux:  /path/to/Slicer/lib/Slicer-5.8/cli-modules/BRAINSFit
BRAINSFIT = "/path/to/BRAINSFit"
BRAINSRESAMPLE = "/path/to/BRAINSResample"

print(f"Input:   {INPUT_DIR}")
print(f"Output:  {OUTPUT_DIR}")
print(f"Modality: {MODALITY}")

---

## Option A: CLI Usage

The simplest way to run the pipeline. Each command below can also be run directly in a terminal (without the `!` prefix).

### Full pipeline in one command

This runs all three steps (reorientation, skull-stripping, registration & defacing) and creates three subdirectories under the output directory:
- `reoriented/` — reoriented scans
- `skullstripped/` — brain-extracted scans and masks
- `defaced/` — final defaced scans

In [ ]:
!caideface run "{INPUT_DIR}" "{OUTPUT_DIR}" \
  --modality {MODALITY} \
  --brainsfit "{BRAINSFIT}" \
  --brainsresample "{BRAINSRESAMPLE}"

### Running individual steps

You can also run each step separately for more control.

#### Step 1: Reorientation

Aligns NIfTI scans to RAS canonical orientation using nibabel (equivalent to FSL's `fslreorient2std`). This standardises the spatial layout, which is required for using HD-BET in the next step.

In [ ]:
REORIENTED_DIR = os.path.join(OUTPUT_DIR, "reoriented")

!caideface reorient "{INPUT_DIR}" "{REORIENTED_DIR}"

#### Step 2: Skull-stripping

Extracts a brain mask and applies dynamic dilation to preserve peripheral brain structures.

- **MRI**: Uses [HD-BET](https://github.com/MIC-DKFZ/HD-BET) for brain extraction.
- **CT**: Uses [TotalSegmentator](https://github.com/wasserth/TotalSegmentator) (requires `pip install caideface[ct]`).

**Why dilation?** Dilation ensures that all brain structures, including peripheral regions and tumours near the skull base, are retained. The structuring element is computed dynamically in millimetres and converted to voxels, so that dilation covers a consistent physical distance (default 14 mm) regardless of voxel resolution.

In [ ]:
SKULLSTRIPPED_DIR = os.path.join(OUTPUT_DIR, "skullstripped")

!caideface skull-strip "{REORIENTED_DIR}" "{SKULLSTRIPPED_DIR}" \
  --modality {MODALITY} \
  --device cpu

#### Step 3: Registration & Defacing

Registers each dilated skull-stripped scan to a modality-matched template using BRAINSFit (affine), warps the template face mask into the scan's space, and applies it to remove facial features.

If a pre-computed transform (`Transform_to_template.txt`) exists alongside a scan, the pipeline will use it instead of running BRAINSFit. This is useful for cases where automatic registration failed and a manual transform was created in 3D Slicer.

In [ ]:
DEFACED_DIR = os.path.join(OUTPUT_DIR, "defaced")

!caideface deface "{REORIENTED_DIR}" "{SKULLSTRIPPED_DIR}" "{DEFACED_DIR}" \
  --modality {MODALITY} \
  --brainsfit "{BRAINSFIT}" \
  --brainsresample "{BRAINSRESAMPLE}"

---

## Option B: Python API

For programmatic use or tighter integration with your own scripts.

### Full pipeline

In [ ]:
from caideface import DefacePipeline

pipeline = DefacePipeline(
    brainsfit_path=BRAINSFIT,
    brainsresample_path=BRAINSRESAMPLE,
    modality=MODALITY,
    device="cpu",
)

results = pipeline.run(INPUT_DIR, OUTPUT_DIR)

failed = results.get("failed_defacing", [])
if failed:
    print(f"{len(failed)} scan(s) failed to deface.")
else:
    print("All scans defaced successfully.")

### Individual steps

In [ ]:
from caideface import reorient_batch, skull_strip_batch, deface_batch

REORIENTED_DIR = os.path.join(OUTPUT_DIR, "reoriented")
SKULLSTRIPPED_DIR = os.path.join(OUTPUT_DIR, "skullstripped")
DEFACED_DIR = os.path.join(OUTPUT_DIR, "defaced")

# Step 1: Reorientation
reorient_log = reorient_batch(INPUT_DIR, REORIENTED_DIR)
print(f"Reoriented {len(reorient_log)} scans")

# Step 2: Skull-stripping
ss_log = skull_strip_batch(
    input_dir=REORIENTED_DIR,
    output_dir=SKULLSTRIPPED_DIR,
    modality=MODALITY,
    device="cpu",
)
print(f"Skull-stripped {len(ss_log)} scans")

# Step 3: Registration & Defacing
failed = deface_batch(
    reoriented_dir=REORIENTED_DIR,
    skullstripped_dir=SKULLSTRIPPED_DIR,
    output_dir=DEFACED_DIR,
    brainsfit_path=BRAINSFIT,
    brainsresample_path=BRAINSRESAMPLE,
    modality=MODALITY,
)
print(f"{len(failed)} scan(s) failed" if failed else "All scans defaced successfully.")

---

## Text Anonymisation

`caideface` also includes a text anonymisation module that detects personal names in medical reports using a trained spaCy NER model and replaces them with realistic fake names, a technique known as Hiding in Plain Sight (HIPS). This anonymisation process is described in 'Evaluation of Named Entity Recognition for Automated Extraction of Present Tumor Size and Personal Names from Radiology Reports Using Spacy' ([doi.org/10.1055/s-0045-1803715](https://doi.org/10.1055/s-0045-1803715)).

### CLI

In [ ]:
# Batch: anonymise all .txt files in a directory
# !caideface anonymize ./reports ./anonymized_reports --seed 42

# Single file
# !caideface anonymize-single ./reports/report_1.txt ./anonymized/report_1.txt

### Python API

In [ ]:
# from caideface import anonymize_batch
#
# log_df = anonymize_batch(
#     input_dir="./reports",
#     output_dir="./anonymized_reports",
#     seed=42,
# )
# print(log_df)

---

## Inspect Results

Load and visualise a defaced scan to verify the output.

In [ ]:
import nibabel as nib
import matplotlib.pyplot as plt
from glob import glob

defaced_files = sorted(glob(os.path.join(OUTPUT_DIR, "defaced", "**", "*_masked.nii.gz"), recursive=True))

if defaced_files:
    img = nib.load(defaced_files[0])
    data = img.get_fdata()
    vs = img.header.get_zooms()[:3]  # voxel sizes in mm (X, Y, Z)

    mid = [s // 2 for s in data.shape[:3]]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    # Sagittal: data[x, Y, Z].T -> rows=Z, cols=Y
    axes[0].imshow(data[mid[0], :, :].T, cmap="gray", origin="lower",
                   aspect=vs[2] / vs[1])
    axes[0].set_title("Sagittal")
    # Coronal: data[X, y, Z].T -> rows=Z, cols=X
    axes[1].imshow(data[:, mid[1], :].T, cmap="gray", origin="lower",
                   aspect=vs[2] / vs[0])
    axes[1].set_title("Coronal")
    # Axial: data[X, Y, z].T -> rows=Y, cols=X
    axes[2].imshow(data[:, :, mid[2]].T, cmap="gray", origin="lower",
                   aspect=vs[1] / vs[0])
    axes[2].set_title("Axial")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(os.path.basename(defaced_files[0]), fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("No defaced scans found. Run the pipeline first.")